# 🎾 OMNIS-COURT LLM + Jina Server (v7.6 ULTIMATE)
## Qwen3-8B + 128K Context + Category 2 KV Cache Optimization

**Instructions:**
1. Runtime → Factory reset runtime
2. Runtime → Change runtime type → **T4 GPU**
3. Run All (Ctrl+F9)
4. Wait ~8-10 minutes
5. Copy both URLs from Cell 5
6. Paste into config/platforms.json
7. Close tab (anti-idle active)

**Improvements:**
- ✅ Qwen3-8B (faster load: 8-10 min vs 30 min)
- ✅ 128K context window (vs 32K)
- ✅ KV Cache optimization (PagedAttention + Prefix Caching + Chunked Prefill)
- ✅ vLLM built-in server (2-3x faster inference)

In [ ]:
# ==========================================
# CELL 1: INSTALL DEPENDENCIES
# ==========================================
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

# Uninstall any vLLM leftovers
!pip uninstall -y vllm flashinfer-python humming-kernels -q 2>/dev/null

# Core packages
!pip install -q bitsandbytes accelerate trafilatura fastapi uvicorn nest-asyncio requests

# vLLM (for optimized inference + KV Cache)
!pip install -q vllm

# Install cloudflared binary
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify installations
import subprocess
cf = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'\n✅ cloudflared: {cf.stdout.strip()}')

all_ok = True
for pkg in ['torch', 'transformers', 'bitsandbytes', 'accelerate', 'trafilatura', 'fastapi', 'uvicorn', 'vllm']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'✅ {pkg}')
    except Exception as e:
        print(f'❌ {pkg}: {e}')
        all_ok = False

if all_ok:
    print('\n✅ ALL dependencies ready!')
    print('⚠️  NOW: Runtime → Restart runtime → Then Run All again')
else:
    print('\n❌ SOME packages failed. STOP here and send error.')

In [ ]:
# ==========================================
# CELL 2: ANTI-IDLE
# ==========================================
from IPython.display import display, Javascript

display(Javascript('''
    setInterval(function(){
        var btn = document.querySelector('colab-run-button');
        if(btn) btn.click();
    }, 300000);
'''))
print('✅ Anti-idle active! Safe to close tab after all cells run.')

In [ ]:
# ==========================================
# CELL 3: LOAD QWEN3-8B + vLLM SERVER
# Qwen3-8B + 128K Context + Category 2 KV Cache Optimization
# ==========================================
import subprocess
import time
import requests

print('🚀 Loading Qwen3-8B with vLLM + Category 2 KV Cache Optimization...')
print('   - Model: Qwen3-8B (4-bit quantization)')
print('   - Context: 128K tokens')
print('   - KV Cache: PagedAttention + Prefix Caching + Chunked Prefill')
print('   - Expected load time: 8-10 minutes')
print()

# vLLM command with ALL Category 2 optimizations
vllm_cmd = """
python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen3-8B \
    --served-model-name qwen3 \
    --port 8000 \
    --host 0.0.0.0 \
    --quantization bitsandbytes \
    --load-format bitsandbytes \
    --dtype auto \
    --trust-remote-code \
    --max-model-len 131072 \
    --gpu-memory-utilization 0.95 \
    --max-num-seqs 128 \
    --block-size 16 \
    --enable-prefix-caching \
    --enable-chunked-prefill \
    --max-num-batched-tokens 8192 \
    --enforce-eager \
    --disable-log-requests
"""

# Start vLLM server in background
process = subprocess.Popen(
    vllm_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for server to be ready
print('⏳ Waiting for vLLM server to start...')
start_time = time.time()
for i in range(180):  # Wait up to 3 minutes for initial startup
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            elapsed = time.time() - start_time
            print(f'\n✅ vLLM Server READY on port 8000 ({elapsed:.0f}s)')
            print('\n📊 Server Configuration:')
            print('   - Model: Qwen3-8B (4-bit)')
            print('   - Context: 128K tokens')
            print('   - KV Cache: Optimized (PagedAttention + Prefix Caching + Chunked Prefill)')
            print('   - GPU Memory Utilization: 95%')
            break
    except:
        pass
    time.sleep(2)
    if i % 15 == 0 and i > 0:
        print(f'   Still loading... ({i*2}s elapsed)')
else:
    print('❌ vLLM Server failed to start within 6 minutes')
    print('\n📋 Last 50 lines of output:')
    stdout, stderr = process.communicate()
    print(stderr[-3000:] if stderr else 'No error output')

In [ ]:
# ==========================================
# CELL 4: START JINA READER SERVER
# ==========================================
import threading, time, requests as req
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn

jina_app = FastAPI(title='OMNIS Jina Reader')

@jina_app.get('/health')
async def jina_health():
    return {'status':'ok'}

@jina_app.get('/extract')
async def extract(url: str = Query(...)):
    try:
        dl = trafilatura.fetch_url(url)
        if not dl:
            return JSONResponse(400, content={'error':'fetch failed','url':url})
        txt = trafilatura.extract(dl, include_comments=False, include_tables=True, no_fallback=False)
        if not txt or len(txt.strip()) < 50:
            return JSONResponse(400, content={'error':'content too short','url':url})
        return {'url':url,'content':txt,'word_count':len(txt.split()),'status':'success'}
    except Exception as e:
        return JSONResponse(500, content={'error':str(e),'url':url})

def run_jina():
    uvicorn.run(jina_app, host='0.0.0.0', port=8001, log_level='warning')

jina_thread = threading.Thread(target=run_jina, daemon=True)
jina_thread.start()
time.sleep(3)

try:
    r = req.get('http://localhost:8001/health', timeout=5)
    print('✅ Jina Reader READY on port 8001' if r.status_code==200 else '❌ Jina error')
except Exception as e:
    print(f'❌ Jina failed: {e}')

In [ ]:
# ==========================================
# CELL 5: CLOUDFLARE TUNNELS
# ==========================================
import subprocess, re

def tunnel(port):
    p = subprocess.Popen(
        ['cloudflared','tunnel','--url',f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return p, m.group(0)
    return p, None

print('🌐 Tunnel LLM (8000)...')
p1, u1 = tunnel(8000)
print('🌐 Tunnel Jina (8001)...')
p2, u2 = tunnel(8001)

if u1 and u2:
    print('\n' + '='*60)
    print('🎉 OMNIS-COURT COLAB READY! (v7.6 ULTIMATE)')
    print('='*60)
    print(f'🧠 LLM:  {u1}')
    print(f'📖 JINA: {u2}')
    print('='*60)
    print('📋 COPY BOTH URLs → config/platforms.json')
    print('🔒 Anti-idle ON → safe to close tab')
    print('🧪 Test URLs from YOUR browser (not from Colab)')
    print('🚀 Features: 128K context + KV Cache optimization')
    print('='*60)
else:
    print(f'❌ Tunnel failed: LLM={u1}, Jina={u2}')

In [ ]:
# ==========================================
# CELL 6: LOCALHOST TESTS
# ==========================================
import requests

print('🧪 Testing LLM on localhost:8000...')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={'model':'qwen3','messages':[{'role':'user','content':'Say OK if you are Qwen3-8B with 128K context'}],'max_tokens':50,'temperature':0.7},
        timeout=120
    )
    if r.status_code == 200:
        resp = r.json()['choices'][0]['message']['content']
        print(f'✅ LLM localhost OK: {resp[:100]}')
    else:
        print(f'❌ LLM localhost: {r.status_code} - {r.text[:200]}')
except Exception as e:
    print(f'❌ LLM localhost: {e}')

print('\n🧪 Testing Jina on localhost:8001...')
try:
    r = requests.get(
        'http://localhost:8001/extract',
        params={'url':'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ Jina localhost OK: {r.json()['word_count']} words")
    else:
        print(f'❌ Jina localhost: {r.status_code}')
except Exception as e:
    print(f'❌ Jina localhost: {e}')

print('\n' + '='*60)
print('🌐 NOW TEST TUNNEL URLs FROM YOUR BROWSER:')
print(f'   {u1}/v1/models')
print(f'   {u2}/health')
print('='*60)
print('\n📊 Expected Performance:')
print('   - Load time: 8-10 minutes (vs 30 min before)')
print('   - Context: 128K tokens (vs 32K before)')
print('   - Inference speed: 2-3x faster (vLLM optimized)')
print('   - VRAM usage: ~5GB (vs 9.3GB before)')
print('='*60)